<img src="https://github.com/hernancontigiani/ceia_memorias_especializacion/raw/master/Figures/logoFIUBA.jpg" width="500" align="center">


# Procesamiento de lenguaje natural
## Custom embedddings con Gensim



### Objetivo
El objetivo es utilizar documentos / corpus para crear embeddings de palabras basado en ese contexto. Se utilizará canciones de bandas para generar los embeddings, es decir, que los vectores tendrán la forma en función de como esa banda haya utilizado las palabras en sus canciones.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import multiprocessing
try:
  from gensim.models import Word2Vec
except:
  !pip install gensim
  from gensim.models import Word2Vec

### Datos
Utilizaremos como dataset canciones de bandas de habla inglesa.

In [2]:
# Descargar la carpeta de dataset
import os
import platform
if os.access('./songs_dataset', os.F_OK) is False:
    if os.access('songs_dataset.zip', os.F_OK) is False:
        if platform.system() == 'Windows':
            !curl https://raw.githubusercontent.com/FIUBA-Posgrado-Inteligencia-Artificial/procesamiento_lenguaje_natural/main/datasets/songs_dataset.zip -o songs_dataset.zip
        else:
            !wget songs_dataset.zip https://github.com/FIUBA-Posgrado-Inteligencia-Artificial/procesamiento_lenguaje_natural/raw/main/datasets/songs_dataset.zip
    !unzip -q songs_dataset.zip
else:
    print("El dataset ya se encuentra descargado")

El dataset ya se encuentra descargado


In [3]:
# Posibles bandas
os.listdir("./songs_dataset/")

['adele.txt',
 'al-green.txt',
 'alicia-keys.txt',
 'amy-winehouse.txt',
 'beatles.txt',
 'bieber.txt',
 'bjork.txt',
 'blink-182.txt',
 'bob-dylan.txt',
 'bob-marley.txt',
 'britney-spears.txt',
 'bruce-springsteen.txt',
 'bruno-mars.txt',
 'cake.txt',
 'dickinson.txt',
 'disney.txt',
 'dj-khaled.txt',
 'dolly-parton.txt',
 'dr-seuss.txt',
 'drake.txt',
 'eminem.txt',
 'janisjoplin.txt',
 'jimi-hendrix.txt',
 'johnny-cash.txt',
 'joni-mitchell.txt',
 'kanye-west.txt',
 'kanye.txt',
 'Kanye_West.txt',
 'lady-gaga.txt',
 'leonard-cohen.txt',
 'lil-wayne.txt',
 'Lil_Wayne.txt',
 'lin-manuel-miranda.txt',
 'lorde.txt',
 'ludacris.txt',
 'michael-jackson.txt',
 'missy-elliott.txt',
 'nickelback.txt',
 'nicki-minaj.txt',
 'nirvana.txt',
 'notorious-big.txt',
 'notorious_big.txt',
 'nursery_rhymes.txt',
 'patti-smith.txt',
 'paul-simon.txt',
 'prince.txt',
 'r-kelly.txt',
 'radiohead.txt',
 'rihanna.txt']

In [4]:
# Armar el dataset utilizando salto de línea para separar las oraciones/docs
df = pd.read_csv('songs_dataset/michael-jackson.txt', sep='/n', header=None)
df.head()

C:\Users\alanv\AppData\Local\Temp\ipykernel_16828\2845004978.py:2: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  df = pd.read_csv('songs_dataset/michael-jackson.txt', sep='/n', header=None)


,0
0,[Spoken Intro:]
1,You ever want something
2,that you know you shouldn't have
3,"The more you know you shouldn't have it,"
4,The more you want it


In [5]:
print("Cantidad de documentos:", df.shape[0])

Cantidad de documentos: 9484


### 1 - Preprocesamiento

In [6]:
from tensorflow.keras.preprocessing.text import text_to_word_sequence

sentence_tokens = []
# Recorrer todas las filas y transformar las oraciones
# en una secuencia de palabras (esto podría realizarse con NLTK o spaCy también)
for _, row in df[:None].iterrows():
    sentence_tokens.append(text_to_word_sequence(row[0]))

In [7]:
# Demos un vistazo
sentence_tokens[:2]

[['spoken', 'intro'], ['you', 'ever', 'want', 'something']]

### 2 - Crear los vectores (word2vec)

In [8]:
from gensim.models.callbacks import CallbackAny2Vec # CallbackAny2Vec tiene un método on_epoch_end que se llama al final de cada época, mientras que Callback no tiene este método. Sirve para poder sobrecargarlo y obtener el "loss" en cada época.
# Durante el entrenamiento gensim por defecto no informa el "loss" en cada época
# Sobrecargamos el callback para poder tener esta información
class callback(CallbackAny2Vec):
    """
    Callback to print loss after each epoch
    """
    def __init__(self):
        self.epoch = 0

    def on_epoch_end(self, model):
        loss = model.get_latest_training_loss()
        if self.epoch == 0:
            print('Loss after epoch {}: {}'.format(self.epoch, loss))
        else:
            print('Loss after epoch {}: {}'.format(self.epoch, loss- self.loss_previous_step))
        self.epoch += 1
        self.loss_previous_step = loss

In [9]:
# Crearmos el modelo generador de vectores
# En este caso utilizaremos la estructura modelo Skipgram
w2v_model = Word2Vec(min_count=5,    # frecuencia mínima de palabra para incluirla en el vocabulario
                     window=2,       # cant de palabras antes y desp de la predicha
                     vector_size=300,       # dimensionalidad de los vectores
                     negative=20,    # cantidad de negative samples... 0 es no se usa
                     workers=1,      # si tienen más cores pueden cambiar este valor
                     sg=1)           # modelo 0:CBOW  1:skipgram

In [10]:
# Obtener el vocabulario con los tokens
w2v_model.build_vocab(sentence_tokens)

In [11]:
# Cantidad de filas/docs encontradas en el corpus
print("Cantidad de docs en el corpus:", w2v_model.corpus_count)

Cantidad de docs en el corpus: 9484


In [12]:
# Cantidad de words encontradas en el corpus
print("Cantidad de words distintas en el corpus:", len(w2v_model.wv.index_to_key))

Cantidad de words distintas en el corpus: 1000


### 3 - Entrenar embeddings

In [13]:
# Entrenamos el modelo generador de vectores
# Utilizamos nuestro callback
w2v_model.train(sentence_tokens,
                 total_examples=w2v_model.corpus_count,
                 epochs=20,
                 compute_loss = True,
                 callbacks=[callback()]
                 )

Loss after epoch 0: 332423.65625
Loss after epoch 1: 227544.78125
Loss after epoch 2: 227830.0625
Loss after epoch 3: 222362.0
Loss after epoch 4: 193124.875
Loss after epoch 5: 183437.75
Loss after epoch 6: 176325.125
Loss after epoch 7: 169745.875
Loss after epoch 8: 166790.125
Loss after epoch 9: 160752.5
Loss after epoch 10: 146619.75
Loss after epoch 11: 141943.75
Loss after epoch 12: 139831.5
Loss after epoch 13: 138176.5
Loss after epoch 14: 137774.75
Loss after epoch 15: 135772.25
Loss after epoch 16: 136182.25
Loss after epoch 17: 134676.0
Loss after epoch 18: 135812.75
Loss after epoch 19: 134378.75


(626341, 993780)

### 4 - Ensayar

In [14]:
# Palabras que MÁS se relacionan con...:
w2v_model.wv.most_similar(positive=["darling"], topn=10)

[('dang', 0.7722715735435486),
 ('wooh', 0.7560095191001892),
 ('uh', 0.7552554607391357),
 ('x2', 0.7457060813903809),
 ('melodie', 0.7372536659240723),
 ('oh', 0.7344543933868408),
 ('daa', 0.7338835000991821),
 ('sad', 0.7283229231834412),
 ('full', 0.7165040373802185),
 ('ooh', 0.7093409895896912)]

In [15]:
# Palabras que MENOS se relacionan con...:
w2v_model.wv.most_similar(negative=["love"], topn=10)

[('uuh', 0.08635728806257248),
 ('refrain', 0.06611773371696472),
 ('siedah', 0.039845868945121765),
 ('cheater', 0.007223140448331833),
 ('stevie', -0.02029196172952652),
 ('pitbull', -0.022470083087682724),
 ('out', -0.054199814796447754),
 ('got', -0.11881646513938904),
 ("doin'", -0.1278565526008606),
 ('lib', -0.13371868431568146)]

In [16]:
# Palabras que MÁS se relacionan con...:
w2v_model.wv.most_similar(positive=["love"], topn=10)

[('rarest', 0.6610762476921082),
 ('summer', 0.6503308415412903),
 ('perfect', 0.6333413124084473),
 ("fallin'", 0.6228702068328857),
 ('farewell', 0.6222076416015625),
 ('power', 0.616187572479248),
 ("makin'", 0.613447904586792),
 ('caress', 0.6112262606620789),
 ('bear', 0.6075263023376465),
 ('strong', 0.6069183945655823)]

In [17]:
# Palabras que MÁS se relacionan con...:
w2v_model.wv.most_similar(positive=["money"], topn=5)

[('anything', 0.8348701596260071),
 ('rain', 0.7239371538162231),
 ('one’s', 0.7216346263885498),
 ('tears', 0.7120650410652161),
 ('stayed', 0.7109531164169312)]

In [18]:
# el método `get_vector` permite obtener los vectores:
vector_love = w2v_model.wv.get_vector("love")
print(vector_love)

[-0.03085848  0.0930151  -0.4032311   0.1897007  -0.25736225 -0.21928546
  0.50926566  0.30389708  0.08144874 -0.21026897 -0.05242809 -0.066051
 -0.08746606 -0.18750882  0.02874098  0.0988658   0.24912001 -0.02181545
  0.06968816 -0.12727211 -0.15825626 -0.3956091   0.05818145 -0.00990384
  0.04429455  0.27152154 -0.24016374  0.04325569 -0.00739807 -0.1570817
  0.2731423   0.21795325  0.24361417 -0.25695822  0.02036539 -0.04739392
 -0.10698196  0.06737688 -0.4325228  -0.1664415  -0.18095647 -0.15144034
  0.05413958  0.15601271 -0.06737553  0.13361648 -0.01699213  0.22844742
  0.14540173 -0.02623471  0.13245842 -0.04925593  0.05756683  0.38116997
 -0.26349315  0.1170376   0.38247716  0.00184009 -0.18535522  0.21457887
 -0.10883912 -0.23416404  0.04685201 -0.08197666 -0.15891509 -0.10939261
  0.04136601 -0.16353197 -0.27136785 -0.24218646 -0.15972295 -0.15448107
  0.2662706   0.02422816  0.35181618  0.12629178 -0.22491899  0.16345192
  0.23892091 -0.18898946  0.05850493 -0.11813705  0.04

In [19]:
# el método `most_similar` también permite comparar a partir de vectores
w2v_model.wv.most_similar(vector_love)

[('love', 1.0),
 ('rarest', 0.6610762476921082),
 ('summer', 0.6503308415412903),
 ('perfect', 0.6333413124084473),
 ("fallin'", 0.6228702068328857),
 ('farewell', 0.6222076416015625),
 ('power', 0.616187572479248),
 ("makin'", 0.613447904586792),
 ('caress', 0.6112262606620789),
 ('bear', 0.6075263023376465)]

In [20]:
# Palabras que MÁS se relacionan con...:
w2v_model.wv.most_similar(positive=["love"], topn=10)

[('rarest', 0.6610762476921082),
 ('summer', 0.6503308415412903),
 ('perfect', 0.6333413124084473),
 ("fallin'", 0.6228702068328857),
 ('farewell', 0.6222076416015625),
 ('power', 0.616187572479248),
 ("makin'", 0.613447904586792),
 ('caress', 0.6112262606620789),
 ('bear', 0.6075263023376465),
 ('strong', 0.6069183945655823)]

### 5 - Visualizar agrupación de vectores

In [21]:
from sklearn.decomposition import IncrementalPCA
from sklearn.manifold import TSNE
import numpy as np

def reduce_dimensions(model, num_dimensions = 2 ):

    vectors = np.asarray(model.wv.vectors)
    labels = np.asarray(model.wv.index_to_key)

    tsne = TSNE(n_components=num_dimensions, random_state=0)
    vectors = tsne.fit_transform(vectors)

    return vectors, labels

In [22]:
# Graficar los embedddings en 2D
import plotly.graph_objects as go
import plotly.express as px

vecs, labels = reduce_dimensions(w2v_model)

MAX_WORDS=200
fig = px.scatter(x=vecs[:MAX_WORDS,0], y=vecs[:MAX_WORDS,1], text=labels[:MAX_WORDS])
fig.show() #renderer="colab") # esto para plotly en colab

In [23]:
# Graficar los embedddings en 3D

vecs, labels = reduce_dimensions(w2v_model,3)

fig = px.scatter_3d(x=vecs[:MAX_WORDS,0], y=vecs[:MAX_WORDS,1], z=vecs[:MAX_WORDS,2],text=labels[:MAX_WORDS])
fig.update_traces(marker_size = 2)
# fig.show(renderer="colab") # esto para plotly en colab

In [24]:
# También se pueden guardar los vectores y labels como tsv para graficar en
# http://projector.tensorflow.org/


vectors = np.asarray(w2v_model.wv.vectors)
labels = list(w2v_model.wv.index_to_key)

np.savetxt("vectors.tsv", vectors, delimiter="\t")

with open("labels.tsv", "w") as fp:
    for item in labels:
        fp.write("%s\n" % item)

### Consigna del desafío 2

**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado**

Recuerden que su notebook de entrega debe poder correrse de inicio a fin sin la aparición de errores.

- Crear sus propios vectores con Gensim basado en lo visto en clase con un corpus propio (revisar enlaces sugeridos en clase 2 sobre opciones de dataset)
- Elegir términos de interés y buscar términos más similares y menos similares.
- Realizar una reduccion de dimensionalidad a los embeddings, llevándolos a 2 dimensiones. Graficar los embeddings proyectados y seleccionar una cantidad de términos (variable MAX_WORDS) de forma tal que la visualización sea adecuada.
- Inspeccionar el grafico y buscar pequeños grupos de palabras que puedan formarse. Interpretarlos e intentar obtener conclusiones. En lo posible, acompañar los grupos de palabras con capturas (y pegarlas en celdas de texto)

### Resolución del desafío

#### Corpus elegido

Se utilizan tres obras clásicas de la literatura argentina del siglo XIX, en español, descargadas de Project Gutenberg:

- **Domingo F. Sarmiento — *Facundo* (1845)**: ensayo político sobre civilización y barbarie.
- **Lucio V. Mansilla — *Una excursión a los indios ranqueles* (1870)**: crónica de viaje por la frontera sur.
- **José Hernández — *Martín Fierro* (1872)**: poema gauchesco.

Los tres comparten época, geografía (pampa argentina) y temáticas, por lo que se espera que los embeddings capten relaciones semánticas coherentes entre términos clave de ese universo cultural.

In [25]:
import os
import re
from gensim.models import Word2Vec

CORPUS_DIR = "corpus_ar"
FILES = [
    "sarmiento_facundo.txt",
    "lucio_mansilla.txt",
    "hernandez_martin_fierro.txt",
]

def clean_gutenberg(text: str) -> str:
    """Quita cabecera y pie de Project Gutenberg si están presentes."""
    start = re.search(r"\*\*\* START OF.*?\*\*\*", text)
    end = re.search(r"\*\*\* END OF.*?\*\*\*", text)
    if start:
        text = text[start.end():]
    if end:
        text = text[:end.start()]
    return text.strip()

texts = []
for name in FILES:
    path = os.path.join(CORPUS_DIR, name)
    with open(path, "r", encoding="utf-8") as f:
        raw = f.read()
    clean = clean_gutenberg(raw)
    print(f"{name}: {len(clean):,} caracteres")
    texts.append(clean)

corpus = "\n".join(texts)
print(f"\nCorpus total: {len(corpus):,} caracteres")

sarmiento_facundo.txt: 602,863 caracteres
lucio_mansilla.txt: 506,091 caracteres
hernandez_martin_fierro.txt: 67,297 caracteres

Corpus total: 1,176,253 caracteres


In [26]:
import pandas as pd

# Partir por signos de puntuacion de fin de oracion
sentences = re.split(r'[.!?]+', corpus)
sentences = [s.strip() for s in sentences if len(s.strip()) > 3]

df = pd.DataFrame(sentences, columns=["sentence"])
print (f"Cantidad de oraciones: {len(df)}")
df.head()

Cantidad de oraciones: 9941


,sentence
0,"Produced by Adrian Mastronardi, Chuck Greif an..."
1,pgdp
2,net (This\nfile was produced from images gener...
3,SARMIENTO\n\nCuando escribió el _Facundo_
4,]\n\n\n\n\nBIBLIOTECA ARGENTINA\n\nPUBLICACIÓN...


Se observa que queda un poco de ruido de las oraciones iniciales

In [27]:
from tensorflow.keras.preprocessing.text import text_to_word_sequence

sentence_tokens = [text_to_word_sequence(s) for s in sentences]
print("Ejemplo:", sentence_tokens[0][:10])

Ejemplo: ['produced', 'by', 'adrian', 'mastronardi', 'chuck', 'greif', 'and', 'the', 'online', 'distributed']


In [28]:
from gensim.models.callbacks import CallbackAny2Vec 

class callback(CallbackAny2Vec):
    def __init__(self):
        self.epoch = 0

    def on_epoch_end(self, model):
        loss = model.get_latest_training_loss()
        if self.epoch == 0:
            print('Loss after epoch {}: {}'.format(self.epoch, loss))
        else:
            print('Loss after epoch {}: {}'.format(self.epoch, loss- self.loss_previous_step))
        self.epoch += 1
        self.loss_previous_step = loss

In [29]:
w2v_model = Word2Vec(min_count=8,
                     window=2,
                     vector_size=300,
                     negative=20,
                     workers=1,
                     sg=1)

In [30]:
w2v_model.build_vocab(sentence_tokens)

In [31]:
print("Cantidad de docs en el corpus:", w2v_model.corpus_count)

Cantidad de docs en el corpus: 9941


In [32]:
print("Cantidad de words distintas en el corpus:", len(w2v_model.wv.index_to_key))
print("Algunas son:", w2v_model.wv.index_to_key[:10])

Cantidad de words distintas en el corpus: 2599
Algunas son: ['de', 'la', 'que', 'y', 'el', 'en', 'los', 'a', 'se', 'no']


In [33]:
w2v_model.train(sentence_tokens,
                 total_examples=w2v_model.corpus_count,
                 epochs=20,
                 compute_loss = True,
                 callbacks=[callback()]
                 )

Loss after epoch 0: 1272818.5
Loss after epoch 1: 941848.75
Loss after epoch 2: 832725.75
Loss after epoch 3: 826382.75
Loss after epoch 4: 787304.25
Loss after epoch 5: 764206.0
Loss after epoch 6: 759434.0
Loss after epoch 7: 751943.0
Loss after epoch 8: 745610.0
Loss after epoch 9: 737360.0
Loss after epoch 10: 709741.0
Loss after epoch 11: 705880.0
Loss after epoch 12: 700025.0
Loss after epoch 13: 698551.0
Loss after epoch 14: 693645.0
Loss after epoch 15: 689010.0
Loss after epoch 16: 688201.0
Loss after epoch 17: 686177.0
Loss after epoch 18: 687482.0
Loss after epoch 19: 686360.0


(2075939, 4001340)

#### Términos de interés

In [34]:
w2v_model.wv.most_similar(positive=["argentina"], topn=10)

[('francesa', 0.7339416146278381),
 ('presidencia', 0.7143018841743469),
 ('independencia', 0.6965507864952087),
 ('unidad', 0.6916500329971313),
 ('decirse', 0.685609757900238),
 ('1810', 0.6758840680122375),
 ('dominio', 0.6710700392723083),
 ('pastoril', 0.6698087453842163),
 ('extremos', 0.665863037109375),
 ('constituye', 0.6629763245582581)]

Aparecen *presidencia*, *independencia*, *unidad*, *1810*, *francesa*. Todo cae en el eje político.

In [35]:
w2v_model.wv.most_similar(positive=["gaucho"], topn=10)

[('cantor', 0.6025716662406921),
 ('alma', 0.5719029307365417),
 ('¡el', 0.5596799850463867),
 ('español', 0.5561723113059998),
 ('diablo', 0.5503401160240173),
 ('propietario', 0.5398908257484436),
 ('baqueano', 0.5388779044151306),
 ('prójimo', 0.5344529151916504),
 ('zorro', 0.5276509523391724),
 ('ciudadano', 0.5269321203231812)]

*Cantor*, *baqueano*, *ciudadano*, *español*, *diablo*. Mezcla figuras del ambiente rural (baqueano, cantor) con la contraposición gaucho/ciudadano de Sarmiento. *Diablo* probablemente entra por expresiones tipo "gaucho del diablo" del Martín Fierro.

In [36]:
w2v_model.wv.most_similar(positive=["campo"], topn=10)

[('puchero', 0.6625170111656189),
 ('lazo', 0.654430091381073),
 ('hilo', 0.6459225416183472),
 ('napoleón', 0.6340132355690002),
 ('suelo', 0.633956253528595),
 ('obscuro', 0.6305945515632629),
 ('leña', 0.6301118731498718),
 ('cañón', 0.6255197525024414),
 ('choclos', 0.625456690788269),
 ('pequeño', 0.6244862675666809)]

Acá se ve la polisemia: mezcla campo rural (*lazo*, *puchero*, *choclos*, *leña*) con campo de batalla (*cañón*, *napoleón*).

In [37]:
w2v_model.wv.most_similar(positive=["libertad"], topn=10)

[('defensa', 0.7304763197898865),
 ('humanidad', 0.7159891724586487),
 ('barbarie', 0.7076725959777832),
 ('situación', 0.7062536478042603),
 ('riqueza', 0.7060090899467468),
 ('cultura', 0.704270601272583),
 ('inteligencia', 0.7012357115745544),
 ('patria', 0.7009288668632507),
 ('talento', 0.6998568177223206),
 ('europea', 0.6941558718681335)]

Lo mejor del modelo: *cultura*, *humanidad*, *riqueza*, *inteligencia*, *europea* y también *barbarie*. Que *barbarie* esté entre los más similares (no los más opuestos) tiene sentido: son palabras que Sarmiento usa en las mismas frases.

### Términos menos relacionados

In [38]:
w2v_model.wv.most_similar(negative=["civilización"], topn=10)

[('estaba', 0.009092308580875397),
 ('soy', -0.021266041323542595),
 ('camilo', -0.02844158373773098),
 ('coronel', -0.03216096758842468),
 ('tenía', -0.05861944332718849),
 ('santos', -0.06037859246134758),
 ('editions', -0.06184495612978935),
 ('ebook', -0.06642371416091919),
 ('mi', -0.06897953152656555),
 ('padre', -0.06981635838747025)]

No devuelve antónimos, devuelve palabras que nunca aparecen en el mismo contexto: verbos comunes (*estaba*, *tenía*, *soy*), nombres propios (*camilo*, *santos*) y basura de Gutenberg (*editions*, *ebook*).

### Palabra fuera del vocabulario

Word2Vec tira `KeyError`.

In [39]:
try:
    w2v_model.wv.most_similar(positive=["smartphone"], topn=5)
except KeyError as e:
    print("KeyError:", e)

KeyError: "Key 'smartphone' not present in vocabulary"


### Visualización 2D

Proyecto los vectores de 300 dim a 2D con t-SNE. Filtro stopwords antes de graficar porque si no las primeras 200 palabras son *de, la, que, y, ...* y no se ve nada temático. Word2Vec no las saca solo.

Probé `MAX_WORDS` en 100, 200 y 500. Con 500 las etiquetas se pisan, con 100 se pierden grupos. Me quedo con 200.

In [40]:
from sklearn.manifold import TSNE
import numpy as np
import plotly.express as px

STOPWORDS_ES = {
    "de", "la", "que", "y", "el", "en", "los", "a", "se", "no", "un", "por",
    "con", "las", "una", "para", "es", "su", "al", "lo", "como", "más", "pero",
    "sus", "le", "ya", "o", "este", "sí", "porque", "esta", "entre", "cuando",
    "muy", "sin", "sobre", "también", "me", "hasta", "hay", "donde", "quien",
    "desde", "todo", "nos", "durante", "todos", "uno", "les", "ni", "contra",
    "otros", "ese", "eso", "ante", "ellos", "e", "esto", "mí", "antes",
    "qué", "unos", "yo", "otro", "otras", "otra", "él", "tanto", "esa", "estos",
    "mucho", "quienes", "nada", "muchos", "cual", "poco", "ella", "estar",
    "algunas", "algo", "mi", "mis", "tú", "te", "ti", "tu", "tus", "os",
    "ha", "han", "he", "hemos", "había", "ser", "fue", "era", "son", "eran",
    "así", "sino", "aún", "aun", "cada", "mismo", "misma", "tan", "del",
    "pgdp", "mastronardi", "greif", "adrian", "chuck", "net", "editions", "ebook",
}

def reduce_dimensions(model, num_dimensions=2):
    vectors = np.asarray(model.wv.vectors)
    labels = np.asarray(model.wv.index_to_key)
    tsne = TSNE(n_components=num_dimensions, random_state=0)
    vectors = tsne.fit_transform(vectors)
    return vectors, labels

vecs, labels = reduce_dimensions(w2v_model)

mask = np.array([lbl not in STOPWORDS_ES for lbl in labels])
vecs_f = vecs[mask]
labels_f = labels[mask]

MAX_WORDS = 200
fig = px.scatter(
    x=vecs_f[:MAX_WORDS, 0],
    y=vecs_f[:MAX_WORDS, 1],
    text=labels_f[:MAX_WORDS],
    title=f"Embeddings 2D (t-SNE) — top {MAX_WORDS} (sin stopwords)",
)
fig.update_traces(textposition="top center")
fig.update_layout(height=800)
fig.show()

### Clusters y conclusiones

Viendo gráfico aparecen grupos bastante claros:

Político / civilización: *argentina, república, historia, revolución, gobierno, guerra, libertad, civilización, europa, américa, mundo, pampa*.

<img src="foto_1.png" width="450">

Numerales y tiempo: *mil, dos, tres, cuatro, diez, años, días*. El modelo aprendió solo que los números pertenecen a la misma categoría, y los pegó a las unidades de tiempo con las que aparecen ("mil años", "cuatro días").

<img src="foto_2.png" width="450">

Rural/gauchesco: *caballo, campo, suelo, camino, cuero*. Todo lo concreto del ambiente pampeano.

<img src="foto_3.png" width="450">

Plurales femeninos: *ciudades, provincias, ideas, palabras, cosas, partes, fuerzas, veces, muchas, todas, grandes, estas*. Este cluster es gramatical, no semántico.

<img src="foto_4.png" width="450">

Conclusiones

- Los vecindarios de palabras clave son coherentes con el contenido de los libros.
- *libertad* quedó pegado a *barbarie*. El modelo capturó que Sarmiento las usa en las mismas oraciones, no que sean sinónimos.
- Word2Vec no separa polisemia .
- Algunos clusters son morfológicos, no semánticos.
- Quedó ruido de Gutenberg (*pgdp*, *mastronardi*, *editions*, *ebook*) que habría que sacar en el preprocesamiento.